## 실습 1: 간단한 고객 지원 에이전트 프로토타입 만들기

### 개요

[Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/)를 사용하면 어떤 프레임워크와 모델을 사용하든 AI 에이전트를 대규모로 안전하게 배포하고 운영할 수 있습니다. 이를 통해 프로토타입에서 프로덕션으로 더 빠르게 전환할 수 있습니다.

5개 실습으로 구성된 이 튜토리얼에서는 **고객 지원 에이전트**를 예로 들어 프로토타입부터 프로덕션까지의 전체 과정을 살펴봅니다. 이 예제에서는 [Google ADK (Agent Development Kit)](https://google.github.io/adk-docs/)와 [Amazon Bedrock](https://aws.amazon.com/bedrock/) 모델을 사용하며, 모델 연결에는 [LiteLLM](https://google.github.io/adk-docs/agents/models/litellm/)을 활용합니다. 실제 애플리케이션에서는 원하는 프레임워크와 모델을 선택할 수 있으며, 여기서 다루는 개념은 다른 프레임워크와 모델에도 적용할 수 있습니다.

**워크숍 과정:**
- **실습 1 (현재)**: 에이전트 프로토타입 만들기 - 실제로 작동하는 고객 지원 에이전트 구축
- **실습 2**: 메모리로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3**: Gateway 및 Identity로 확장 - 여러 에이전트에서 도구를 안전하게 공유
- **실습 4**: 프로덕션에 배포 - 관찰 기능과 함께 AgentCore Runtime 사용
- **실습 5**: 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기

첫 번째 실습에서는 고객 지원 에이전트 프로토타입을 구축합니다. 이 프로토타입은 워크숍을 진행하면서 영구 메모리, 공유 도구, 전체 관찰 기능을 갖추고 여러 고객을 지원하는 프로덕션 준비 시스템으로 발전합니다. 에이전트는 다음 로컬 도구를 사용할 수 있습니다.
- **get_return_policy()** - 특정 제품의 반품 정책 조회
- **get_product_info()** - 제품 정보 조회
- **web_search()** - 문제 해결에 도움이 되는 웹 검색
- **get_technical_support()** - Bedrock Knowledge Base 검색

### 실습 1 아키텍처
<div style="text-align:left">
    <img src="images/architecture_lab1_strands.png" width="75%"/>
</div>

*로컬에서 실행되는 간단한 프로토타입입니다. 이후 실습에서는 공유 도구, 영구 메모리, 프로덕션 수준의 관찰 기능을 갖춘 AgentCore 서비스로 마이그레이션합니다.*

### 사전 요구 사항

* 적절한 권한이 있는 **AWS 계정**
* 로컬에 설치된 **Python 3.10 이상**
* 자격 증명이 설정된 **AWS CLI**
* **Amazon Bedrock** 모델 액세스 활성화(예: Anthropic Claude 또는 Amazon Nova)
* 다음 셀에서 설치할 **Google ADK** 및 기타 라이브러리

#### AWS 워크숍 계정을 사용하지 않나요?

자율 학습 방식으로 실습을 진행하는 경우 CloudFormation 스택을 생성하고 배포하기 위해 다음 2단계를 추가로 반드시 수행해야 합니다.

**단계 0.1: [자율 학습 실습 전용]**

아래 셀의 `!aws sts get-caller-identity` 명령 주석을 해제해 Sagemaker Role을 확인합니다. AWS 콘솔에서 IAM으로 이동하여 Sagemaker Role을 검색합니다. 그런 다음 [IAM Policy, AWS managed policies 및 Trust relationships](https://catalog.us-east-1.prod.workshops.aws/workshops/850fcd5c-fd1f-48d7-932c-ad9babede979/en-US/00-prerequisites/02-self-paced#iam-policy-for-bedrock-agentcore-workshop)를 [워크숍 자율 학습 사전 요구 사항](https://catalog.us-east-1.prod.workshops.aws/workshops/850fcd5c-fd1f-48d7-932c-ad9babede979/en-US/00-prerequisites/02-self-paced) 실습의 설명에 따라 추가합니다.

In [ ]:
# 참고: 자율 학습 실습에서만 주석을 해제하고 실행합니다.
!aws sts get-caller-identity 

**단계 0.2: [자율 학습 실습 전용]**

`prereq.sh` 스크립트에 필요한 권한을 모두 갖추었다면 다음 명령을 실행하여 CloudFormation 템플릿을 배포합니다.

In [ ]:
# 참고: 자율 학습 실습에서만 주석을 해제하고 실행합니다.
# !bash scripts/prereq.sh

### 단계 1: 종속성 설치 및 라이브러리 가져오기
시작하기 전에 이 실습에 필요한 종속성을 설치합니다. 일부 종속성 오류가 표시될 수 있지만, 이 워크숍 범위에서는 무시해도 됩니다.

In [ ]:
# 필수 패키지 설치
%pip install -r requirements.txt -q

이제 필요한 라이브러리를 가져오고 boto3 세션을 초기화합니다.

In [ ]:
# 라이브러리 가져오기
import boto3
from boto3.session import Session

from lab_helpers.utils import suppress_warnings

suppress_warnings()

from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

In [ ]:
# boto 세션 가져오기
boto_session = Session()
region = boto_session.region_name

### 단계 2: 사용자 지정 도구 구현

다음으로 고객 지원 에이전트에 제공할 4가지 도구를 구현합니다.

Google ADK에서 도구는 타입 힌트와 docstring이 포함된 Python 함수입니다. ADK는 함수 시그니처와 문서를 사용하여 각 도구의 컨텍스트를 에이전트에 제공합니다.

#### 도구 1: 반품 정책 조회

**목적:** 이 도구는 고객이 제품 카테고리별 반품 정책을 이해할 수 있도록 지원합니다. 반품 가능 기간, 조건, 절차, 환불 처리 기간에 관한 상세 정보를 제공하여 고객이 제품 반품 시 예상되는 과정을 정확히 알 수 있게 합니다.

In [ ]:
def get_return_policy(product_category: str) -> str:
    """Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # 모의 반품 정책 데이터베이스 - 실제 구현에서는 정책 데이터베이스를 조회합니다.
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
    }

    # 목록에 없는 카테고리에 적용할 기본 정책
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection",
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies",
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    return (
        f"Return Policy - {product_category.title()}:\n\n"
        f"• Return window: {policy['window']} from delivery\n"
        f"• Condition: {policy['condition']}\n"
        f"• Process: {policy['process']}\n"
        f"• Refund timeline: {policy['refund_time']}\n"
        f"• Shipping: {policy['shipping']}\n"
        f"• Warranty: {policy['warranty']}"
    )


print("✅ Return policy tool ready")

#### 도구 2: 제품 정보 조회

**목적:** 이 도구는 보증, 구매 가능한 모델, 주요 기능, 배송 정책, 반품 정보를 포함한 종합적인 제품 정보를 고객에게 제공합니다. 고객이 충분한 정보를 바탕으로 구매를 결정하고 구매할 제품을 이해할 수 있도록 지원합니다.

In [ ]:
def get_product_info(product_type: str) -> str:
    """Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    # 모의 제품 카탈로그 - 실제 구현에서는 제품 데이터베이스를 조회합니다.
    products = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32GB RAM, SSD storage, various display sizes",
            "features": "Backlit keyboards, USB-C/Thunderbolt, Wi-Fi 6, Bluetooth 5.0",
            "compatibility": "Windows 11, macOS, Linux support varies by model",
            "support": "Technical support and driver updates included",
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G/4G connectivity, 128GB-1TB storage, multiple camera systems",
            "features": "Wireless charging, water resistance, biometric security",
            "compatibility": "iOS/Android, carrier unlocked options available",
            "support": "Software updates and technical support included",
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless options, noise cancellation, 20Hz-20kHz frequency",
            "features": "Active noise cancellation, touch controls, voice assistant",
            "compatibility": "Bluetooth 5.0+, 3.5mm jack, USB-C charging",
            "support": "Firmware updates via companion app",
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p resolutions, IPS/OLED panels, various sizes",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support",
        },
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team for detailed product information and compatibility requirements."

    return (
        f"Technical Information - {product_type.title()}:\n\n"
        f"• Warranty: {product['warranty']}\n"
        f"• Specifications: {product['specs']}\n"
        f"• Key Features: {product['features']}\n"
        f"• Compatibility: {product['compatibility']}\n"
        f"• Support: {product['support']}"
    )


print("✅ get_product_info tool ready")

#### 도구 3: 웹 검색

**목적:** 이 도구를 통해 고객은 문제 해결 지원이나 제품 추천 등의 정보를 얻을 수 있습니다.

In [ ]:
def web_search(keywords: str, region: str, max_results: int) -> str:
    """Search the web for updated information.

    Args:
        keywords: The search query keywords.
        region: The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results: The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    if not region:
        region = "us-en"
    if not max_results:
        max_results = 5
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"


print("✅ Web search tool ready")

#### 고객 지원 에이전트 - Knowledge Base 통합 단계

##### S3에서 제품 technical_support 파일 다운로드

In [ ]:
import os


def download_files():
    # 계정 및 리전 가져오기
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    region = boto3.Session().region_name
    bucket_name = f"{account_id}-{region}-kb-data-bucket"

    # 로컬 폴더 생성
    os.makedirs("knowledge_base_data", exist_ok=True)

    # 모든 파일 다운로드
    s3 = boto3.client("s3")
    objects = s3.list_objects_v2(Bucket=bucket_name)

    for obj in objects["Contents"]:
        file_name = obj["Key"]
        s3.download_file(bucket_name, file_name, f"knowledge_base_data/{file_name}")
        print(f"Downloaded: {file_name}")

    print("All files saved to: knowledge_base_data/")


# 실행
download_files()

#### Knowledge Base 동기화 작업

##### 에이전트와 통합할 수 있도록 S3의 제품 technical_support 파일을 Knowledge Base와 동기화

In [ ]:
import time

# 파라미터 가져오기
ssm = boto3.client("ssm")
bedrock = boto3.client("bedrock-agent")
s3 = boto3.client("s3")

account_id = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.Session().region_name

kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")["Parameter"]["Value"]
ds_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/data-source-id")["Parameter"]["Value"]

# S3 버킷에서 파일 이름 가져오기
bucket_name = f"{account_id}-{region}-kb-data-bucket"
s3_objects = s3.list_objects_v2(Bucket=bucket_name)
file_names = [obj["Key"] for obj in s3_objects.get("Contents", [])]

# 동기화 작업 시작
response = bedrock.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id, description="Quick sync")

job_id = response["ingestionJob"]["ingestionJobId"]
print("Bedrock knowledge base sync job started, ingesting the data files from s3")

# 완료될 때까지 모니터링
while True:
    job = bedrock.get_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id)["ingestionJob"]

    status = job["status"]

    if status in ["COMPLETE", "FAILED"]:
        break

    time.sleep(10)

# 최종 결과 출력
if status == "COMPLETE":
    file_count = job.get("statistics", {}).get("numberOfDocumentsScanned", 0)
    files_list = ", ".join(file_names)
    print(f"Bedrock knowledge base sync job completed Successfully, ingested {file_count} files")
    print(f"Files ingested: {files_list}")
else:
    print(f"Bedrock knowledge base sync job failed with status: {status}")

#### 도구 4: 기술 지원 받기

**목적:** 이 도구는 전자 제품 문서가 저장된 Knowledge Base에 액세스하여 고객에게 종합적인 기술 지원과 문제 해결 도움을 제공합니다. 상세 설정 가이드, 유지 관리 지침, 문제 해결 단계, 연결 문제 해결 방법, 보증 서비스 정보가 포함됩니다. 고객이 기술 문제를 해결하고, 기기를 올바르게 구성하며, 최적의 제품 성능에 필요한 유지 관리 요구 사항을 이해하도록 지원합니다.

Strands 대신 Google ADK를 사용하므로 Strands의 `retrieve` 도구를 Bedrock Knowledge Base `retrieve` API에 대한 직접 boto3 호출로 대체합니다.

In [ ]:
def get_technical_support(issue_description: str) -> str:
    """Get technical support and troubleshooting guidance by searching the knowledge base.

    Args:
        issue_description: Description of the technical issue or question.

    Returns:
        Relevant technical support documentation and troubleshooting steps.
    """
    try:
        # Parameter Store에서 KB ID 가져오기
        ssm_client = boto3.client("ssm")
        account_id = boto3.client("sts").get_caller_identity()["Account"]
        region = boto3.Session().region_name

        kb_id = ssm_client.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")["Parameter"]["Value"]
        print(f"Successfully retrieved KB ID: {kb_id}")

        # boto3 bedrock-agent-runtime을 사용하여 Knowledge Base에서 검색
        bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=region)

        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": issue_description},
            retrievalConfiguration={
                "vectorSearchConfiguration": {
                    "numberOfResults": 3,
                }
            },
        )

        # 결과 추출 및 형식 지정
        results = response.get("retrievalResults", [])
        if not results:
            return "No relevant technical support documentation found for this issue."

        formatted_results = []
        for i, result in enumerate(results, 1):
            content = result.get("content", {}).get("text", "")
            score = result.get("score", 0)
            if score >= 0.4:
                formatted_results.append(f"--- Result {i} (relevance: {score:.2f}) ---\n{content}")

        if not formatted_results:
            return "No sufficiently relevant technical support documentation found. Please contact our support team directly."

        return "\n\n".join(formatted_results)

    except Exception as e:
        print(f"Detailed error in get_technical_support: {str(e)}")
        return f"Unable to access technical support documentation. Error: {str(e)}"


print("✅ Technical support tool ready")

### 단계 4: 고객 지원 에이전트 생성 및 구성

다음으로 LiteLLM을 통해 Amazon Bedrock 모델을 사용하는 Google ADK의 `LlmAgent`로 고객 지원 에이전트를 생성합니다. 모델, 이전 단계에서 구현한 도구 목록, 시스템 지침을 제공합니다.

또한 비동기 ADK Runner 호출 패턴을 처리하는 `call_agent` 헬퍼 함수를 정의합니다.

In [ ]:
SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications, and maintenance questions
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information. 
4. get_technical_support() - For troubleshooting issues, setup guides, maintenance tips, and detailed technical assistance
For any technical problems, setup questions, or maintenance concerns, always use the get_technical_support() tool as it contains our comprehensive technical documentation and step-by-step guides.

Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""

APP_NAME = "customer_support_agent"
USER_ID = "user_123"
SESSION_ID = "lab01_session"

# LiteLLM을 통해 Amazon Bedrock을 사용하는 Google ADK 에이전트 생성
agent = LlmAgent(
    model=LiteLlm(model="bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0"),
    name="customer_support_agent",
    description="A customer support agent for an electronics e-commerce company.",
    instruction=SYSTEM_PROMPT,
    tools=[
        get_product_info,  # 도구 1: 간단한 제품 정보 조회
        get_return_policy,  # 도구 2: 간단한 반품 정책 조회
        web_search,  # 도구 3: 최신 정보를 얻기 위한 웹 액세스
        get_technical_support,  # 도구 4: 기술 지원 및 문제 해결
    ],
)


async def call_agent(query: str) -> str:
    """사용자 질의로 ADK 에이전트를 호출하고 응답을 반환합니다."""
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=query)])
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)
    final_response = ""
    async for event in events:
        if event.is_final_response():
            final_response = event.content.parts[0].text
    return final_response


print("Customer Support Agent created successfully!")

### 단계 5: 고객 지원 에이전트 테스트

모든 도구가 올바르게 작동하는지 샘플 질의로 에이전트를 테스트합니다.

#### 반품 확인 테스트

In [ ]:
response = await call_agent("What's the return policy for my thinkpad X1 Carbon?")
print(response)

In [ ]:
response = await call_agent("My laptop won't turn on, what should I check?")
print(response)

#### 문제 해결 테스트

In [ ]:
response = await call_agent("I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?")
print(response)

## 🎉 실습 1 완료!

**LiteLLM**을 통해 **Amazon Bedrock** 모델을 사용하는 **Google ADK**로 실제 작동하는 고객 지원 에이전트 프로토타입을 성공적으로 만들었습니다. 이번 실습에서 완료한 내용은 다음과 같습니다.

- 4가지 도구(반품 정책, 제품 정보, 웹 검색, 기술 지원 KB)를 갖춘 Google ADK 에이전트 구축
- ADK 프레임워크에서 LiteLLM을 사용하여 Amazon Bedrock 모델 호출
- 여러 도구 간 상호 작용 및 웹 검색 기능 테스트
- 프로덕션 전환을 위한 기반 마련

### 현재 제한 사항(이후 실습에서 해결합니다)
- **단일 사용자 대화 메모리** - 로컬 대화 세션만 사용하므로 여러 고객에게는 여러 세션이 필요합니다.
- **세션으로 제한된 대화 기록** - 대화에서 장기 메모리나 세션 간 정보를 사용할 수 없습니다.
- **도구 재사용성** - 서로 다른 에이전트에서 도구를 재사용할 수 없습니다.
- **로컬에서만 실행** - 확장할 수 없습니다.
- **Identity** - 사용자 및/또는 에이전트 identity와 액세스 제어가 없습니다.
- **관찰 기능** - 에이전트 동작을 관찰하는 기능이 제한적입니다.
- **기존 API** - 고객 데이터용 기존 엔터프라이즈 API에 액세스할 수 없습니다.

##### 다음 실습: [실습 2 - 메모리를 추가하여 에이전트 개인화하기 →](lab-02-agentcore-memory.ipynb)